# FigureSI_ — Signal-to-Background Ratio (SBR) Sweep

*Makes up Figures S34–S36.*

Simulates localisation and spectral precision for ATTO 488, ATTO 565, and ATTO 647N
across a grid of signal levels (N photons) and SBR values (100 → 1.1).

SBR is defined as SBR = (I_sig + I_bkg) / I_bkg, where I_sig = N_photons is the total
signal in the punctum and I_bkg is the total background *within the same footprint*
— not a bare per-pixel background level. Per photon level, `SimulationConfig.sbr`
computes the total background photons within the PSF's 3σ circular footprint as
`I_bkg = N_photons / (SBR - 1)`, then converts to the per-pixel `background_photons`
rate the simulator uses by dividing by that footprint's area (`π·(3σ_psf)²`, with
σ_psf from the dye's average emission wavelength and the objective NA). This keeps
SBR comparable across dyes (different σ_psf) and across the photon sweep, rather
than conflating "photons per punctum" with "photons per pixel" the way a bare
`N_photons / SBR` would. SBR = 1 is a singularity (zero signal) under this
definition, so the sweep floor is 1.1 rather than 1.

**Key result**: signal-limited (low N, high SBR) and background-limited (high N,
low SBR) conditions that share the same *effective* photon budget produce similar
distributions in the colour simplex — shown on ternary plots at the end.

**`FAST_MODE` flag:** `True` (default) sweeps 12 photon levels x 6 SBR levels x 150
bootstrap fits per condition instead of the published 200 levels x 25 SBR x 10,000
bootstrap fits — same 3 dyes, just coarser and noisier, in well under a minute.
`FAST_MODE = False` reproduces the published sweep exactly (can take hours).

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s — %(levelname)s — %(message)s')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import polars as pl
import pandas as pd
import os
import time
import glob
import types
from pathlib import Path
from scipy.spatial.distance import cdist
import xarray as xr
import sys

from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, sCMOSFunctions, PlottingBase
from pyS3M.simulation.multicolour import FittingStrategy, SimulationConfig, MultiC_Sim_Funcs

# _sbr.py (the published S34-S36 layouts) lives in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _sbr

IO      = IOFunctions.IO_Functions()
S_F     = SpectralFunctions.Spectral_Funcs()
M_F     = MaskFunctions.Mask_Functions()
sCMOS   = sCMOSFunctions.sCMOS_Functions()
MSF     = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter()

smoothing_function = types.SimpleNamespace(
    args={'sigma': 1.5},
    extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack,
    data_arg='image',
)

REPO_ROOT = Path.cwd().resolve().parents[2]
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT = Path("outputs") / "SBR"  # git-ignored
FIGURE_DIR = Path("figures")        # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FAST_MODE = True  # False -> the published 200-photon-level x 25-SBR x 10,000-bootstrap sweep
print(f"FAST_MODE={FAST_MODE}")

In [ ]:
gain      = IO.read_tiff(str(CALIBRATION_DIR / 'gain.tif'))
offset    = IO.read_tiff(str(CALIBRATION_DIR / 'offset.tif'))
variance  = IO.read_tiff(str(CALIBRATION_DIR / 'variance.tif'))
readnoise = float(np.median(IO.read_tiff(str(CALIBRATION_DIR / 'readnoise.tif'))))
rqe       = IO.read_tiff(str(CALIBRATION_DIR / 'rqe.tif'))
print(f'Camera calibration loaded — sensor: {gain.shape}, readnoise: {readnoise:.2f}')

In [ ]:
# ── Bayer (RGGB) spectral setup ───────────────────────────────────────────────
R_base, G_base, B_base, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_base, G_base, R_base])   # B=0, G=1, R=2
mosaic    = M_F.mosaic_unit                         # default BGGR Ximea pattern

pixel_order         = ['B', 'G', 'R']
pixel_order_indices = {'B': 0, 'G': 1, 'R': 2}

dyes    = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
filters = []
NA          = 1.49
pixel_size  = 69.0
image_size  = 14    # seed size; test_simulation_method resizes via mosaic_unit

print(f'Wavelength: {wavelength[0]:.0f}–{wavelength[-1]:.0f} nm  ({len(wavelength)} pts)')
print(f'Mosaic unit: {mosaic}')

In [ ]:
# ── Save folder + sweep parameters ────────────────────────────────────────────
save_folder = OUT_ROOT

N_PHOTON_STEPS = 12 if FAST_MODE else 200
N_SBR_STEPS = 6 if FAST_MODE else 25
N_BOOTSTRAP = 150 if FAST_MODE else 10_000

n_photon_space = np.unique(
    np.around(np.logspace(np.log10(500), np.log10(50000), N_PHOTON_STEPS) / 5) * 5
)
# Floor is 1.1, not 1: SBR = (I_sig + I_bkg)/I_bkg = 1 implies zero signal, which is
# a singularity for the per-pixel background conversion in SimulationConfig.sbr
# (see markdown above).
sbr_space = np.geomspace(1.1, 100, N_SBR_STEPS)[::-1]


# Filename flag for a given SBR value — shared by the sweep, loader, and
# case-study cells so the saved/looked-up filenames always agree.
def sbr_flag(sbr):
    return f'sbr{sbr:.2f}_'.replace('.', 'p')


config_base = SimulationConfig(
    n_bootstrap=N_BOOTSTRAP,
    background_photons=5.0,   # fallback only; overridden by sbr
    NA=NA,
    pixel_size=pixel_size,
    save_raw_results=True,
    subtractx0y0=False,
    saverawimages=False,
    verbose=False,
    use_stochastic_photons=True,
    n_unit_cells=7,
)

print(f'Photon levels : {len(n_photon_space)}  ({n_photon_space[0]:.0f}–{n_photon_space[-1]:.0f})')
print(f'SBR values    : {sbr_space}')
print(f'Total runs    : {len(dyes)} dyes × {len(sbr_space)} SBRs = {len(dyes)*len(sbr_space)}')

In [ ]:
# ── Camera parameters (mosaic_unit triggers auto-resize in test_simulation_method) ──
camera_params_dict = {
    'gain':               np.full((image_size, image_size), np.median(gain)),
    'offset':             np.full((image_size, image_size), np.median(offset)),
    'variance':           np.full((image_size, image_size), np.median(variance)),
    'readnoise':          readnoise,
    'rqe':                np.full((image_size, image_size), np.median(rqe)),
    'masks':              M_F.get_masks(size_x=image_size, size_y=image_size),
    'pixel_QYs':          pixel_QYs,
    'pixel_order':        pixel_order,
    'pixel_order_indices': pixel_order_indices,
    'mosaic_unit':        mosaic,
}
print(f'Camera params ready — base image size: {image_size}×{image_size} px')

## Simulation

For each dye × SBR, `test_simulation_method` sweeps `n_photon_space`.
Background photons for each frame = `N_photons / SBR` (via `SimulationConfig.sbr`).

In [ ]:
# ── Run SBR sweep ──────────────────────────────────────────────────────────────
n_total = len(dyes) * len(sbr_space)
i_run   = 0
t_start = time.time()

for dye in dyes:
    for sbr in sbr_space:
        i_run += 1
        flag = sbr_flag(sbr)

        config = SimulationConfig(
            n_bootstrap=config_base.n_bootstrap,
            background_photons=config_base.background_photons,
            NA=config_base.NA,
            pixel_size=config_base.pixel_size,
            save_raw_results=config_base.save_raw_results,
            subtractx0y0=config_base.subtractx0y0,
            saverawimages=config_base.saverawimages,
            verbose=config_base.verbose,
            use_stochastic_photons=config_base.use_stochastic_photons,
            n_unit_cells=config_base.n_unit_cells,
            sbr=sbr,
        )

        print(f'[{i_run:2d}/{n_total}]  {dye}  SBR={sbr:.0f}', end='  ', flush=True)

        MSF.test_simulation_method(
            dye=dye,
            filters=filters,
            wavelength=wavelength,
            camera_parameters=camera_params_dict,
            save_folder=str(save_folder),
            n_photon_space=n_photon_space,
            smoothing_function=smoothing_function,
            strategy=FittingStrategy.STANDARD_DATA,
            starting_flag=flag,
            config=config,
            overwrite=True,
        )

        elapsed = (time.time() - t_start) / 60
        rate    = i_run / elapsed if elapsed > 0 else 0
        eta     = (n_total - i_run) / rate if rate > 0 else 0
        print(f'done  ({elapsed:.1f} min, ETA {eta:.1f} min)')

print(f'\nSweep complete in {(time.time()-t_start)/60:.1f} min.')

## Analysis

Load raw HDF5 results and ground-truth CSVs; compute σ_xy (nm) and σ_colour per
(dye, SBR, N_photons) cell.

In [ ]:
# ── Build xarray: [dye, sbr, n_photons, metric] ───────────────────────────────
metrics    = ['sigma_xy', 'sigma_colour']
a_cols     = [f'A_{l}' for l in pixel_order]
n_unit     = config_base.n_unit_cells

SBR_XR = xr.DataArray(
    data=np.full(
        [len(dyes), len(sbr_space), len(n_photon_space), len(metrics)], np.nan
    ),
    coords={
        'dye':       dyes,
        'sbr':       sbr_space,
        'n_photons': n_photon_space,
        'metric':    metrics,
    },
    dims=['dye', 'sbr', 'n_photons', 'metric'],
)

missing = []
for i_d, dye in enumerate(dyes):
    dye_str = dye.replace('/', '-')
    for i_s, sbr in enumerate(sbr_space):
        flag = sbr_flag(sbr)

        h5_path = save_folder / f'{flag}LM_method_{dye_str}_rawresults.h5'
        gt_path = sorted(glob.glob(str(save_folder / f'{flag}*{dye_str}*groundtruth*.csv')))
        ip_path = sorted(glob.glob(str(save_folder / f'{flag}*{dye_str}*input_parameters*.csv')))

        if not h5_path.exists() or not gt_path or not ip_path:
            missing.append((dye, sbr))
            continue

        gt    = pd.read_csv(gt_path[0])
        gt_px = np.vstack([gt['x0'].to_numpy() / pixel_size,
                           gt['y0'].to_numpy() / pixel_size]).T

        ip_row      = pd.read_csv(ip_path[0]).to_numpy()[0]
        colour_raw  = ip_row[-3:]          # B, G, R fractions from input_parameters
        colour_true = np.expand_dims(colour_raw / colour_raw.sum(), 0)

        df_raw = pd.read_hdf(str(h5_path), key='data')

        for k in range(len(n_photon_space)):
            sub  = df_raw[df_raw['photon_level'] == k]
            filt = (
                (sub['xc'].to_numpy() > 0) &
                (sub['yc'].to_numpy() > 0) &
                (sub['s_x'].to_numpy() > 0) &
                (sub['s_y'].to_numpy() > 0) &
                (sub['xc'].to_numpy() < n_unit * mosaic.shape[0]) &
                (sub['yc'].to_numpy() < n_unit * mosaic.shape[1])
            )
            if filt.sum() < 5:
                continue

            ex = (sub['xc'].to_numpy()[filt] - gt_px[filt, 0]) * pixel_size
            ey = (sub['yc'].to_numpy()[filt] - gt_px[filt, 1]) * pixel_size
            SBR_XR[i_d, i_s, k, 0] = np.sqrt(
                (np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2
            )

            colours = np.vstack([sub[c].to_numpy()[filt] for c in a_cols]).T
            SBR_XR[i_d, i_s, k, 1] = float(np.nanstd(cdist(colours, colour_true)))

if missing:
    print(f'Missing: {missing}')
SBR_XR.to_netcdf(str(save_folder / 'SBR_XR.nc'))
print(f'DataArray shape: {SBR_XR.shape}  dims: {list(SBR_XR.dims)}')
print(f'σ_xy  range: {float(np.nanmin(SBR_XR.sel(metric="sigma_xy"))):.1f}–'
      f'{float(np.nanmax(SBR_XR.sel(metric="sigma_xy"))):.1f} nm')
print(f'σ_col range: {float(np.nanmin(SBR_XR.sel(metric="sigma_colour"))):.4f}–'
      f'{float(np.nanmax(SBR_XR.sel(metric="sigma_colour"))):.4f}')

## Precision Surfaces

For each dye, plot a 2D heatmap with N_photons (log x) × SBR (log y)
and colour = precision.  Rows: σ_xy / σ_colour.  Columns: dyes.

In [ ]:
# ── Load (re-run analysis without re-simulating) ──────────────────────────────
SBR_XR = xr.open_dataarray(str(save_folder / 'SBR_XR.nc'))
fs = plotter.config.font_size

In [ ]:
figure_save_folder = str(FIGURE_DIR)

In [ ]:
# ── Surface plots (S35, S36): one panel per dye, as published ─────────────────
for metric in ['sigma_xy', 'sigma_colour']:
    fig = _sbr.surface_figure(SBR_XR, metric, fontsize=fs)
    fname = f'sbr_surface_{metric}.svg'
    plotter.save_or_show(fig, save_path=os.path.join(figure_save_folder, fname), show=True, dpi=600)
    print(f'Saved {fname}')

## Ternary Plots — Case Studies (S34)

Compare three regimes that nominally differ in *kind* of limitation:

| Case | N photons | SBR | Limitation |
|------|-----------|-----|------------|
| A    | ~5 000    | 100 | — (reference) |
| B    | ~500      | 100 | signal-limited |
| C    | ~5 000    | 1.1 | background-limited |

Each panel shows up to 3,000 fits from the sweep point nearest the case's (N, SBR), the dye's true colour as a
white +. If the spread in the colour simplex is similar for B and C, the two regimes are equivalent for spectral
discrimination. Layout as published (`_sbr.ternary_cases_figure`, 466 x 400 pt).

In [ ]:
# ── Case-study fits: the sweep point nearest each case's (N, SBR) ─────────────
rng = np.random.default_rng(0)
samples, truths = [], []
for dye in dyes:
    ip = sorted(glob.glob(str(save_folder / f'{sbr_flag(100.0)}*{dye.replace("/", "-")}*input_parameters*.csv')))
    truth = pd.read_csv(ip[0]).to_numpy()[0][-3:].astype(float) if ip else None   # B, G, R
    truths.append(None if truth is None else truth / truth.sum())
    row = []
    for n_ph, sbr, _ in _sbr.CASES:
        sbr_closest = sbr_space[int(np.argmin(np.abs(sbr_space - sbr)))]
        k_closest = int(np.argmin(np.abs(n_photon_space - n_ph)))
        row.append(_sbr.load_case(save_folder, dye, sbr_flag(sbr_closest), k_closest, n_sub=3000, rng=rng))
        print(f'{dye}: N={n_ph} -> {n_photon_space[k_closest]:.0f}, SBR={sbr} -> {sbr_closest:.2f}, '
              f'{0 if row[-1] is None else len(row[-1])} fits')
    samples.append(row)

In [ ]:
# ── S34: rows = dyes, columns = cases ────────────────────────────────────────
fig = _sbr.ternary_cases_figure(samples, truths, dyes)
for ext in ('svg', 'pdf'):   # fixed page size: no tight bounding box
    plotter.save_or_show(fig, save_path=os.path.join(figure_save_folder, f'sbr_ternary_cases.{ext}'),
                         show=(ext == 'pdf'), dpi=600, bbox_inches=None)
print('Ternary case study figure saved.')